In [ ]:
%pip install -U --quiet numpy>=2.0 pandas>=2.2 scipy>=1.11 scikit-learn>=1.5 shap>=0.46.0 pygad>=3.2.0 pygam>=0.9.0
# interpret (EBM) চাইলে পরে আলাদা সেলে ট্রাই করুন; না মিললে GAM fallback থাকবে।
# %pip install -U --quiet interpret



In [ ]:
# Repo-local run: raw XPT files are expected under data/raw/Dataset.
# Colab drive mounting is not required for this repository version.


In [ ]:
# ================================
# Domain-First Causal Pipeline (Steps 1–3) — Py3.12 + Low-Memory safe
# ================================
import os, re, json, gc, warnings
from pathlib import Path
from typing import Dict, List, Tuple
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=FutureWarning)

# ---------- Low-memory toggles ----------
LOW_MEM          = True
ROW_SUBSAMPLE    = 30000    # max rows to keep for modeling (None to disable)
SHAP_SAMPLE      = 2000     # rows for SHAP explain (smaller -> lighter)
PI_REPEATS       = 5        # permutation repeats (lighter than 8-10)
OHE_MIN_FREQ     = 0.01     # collapse rare categories
OHE_MAX_CATS     = 40       # cap category explosion
GA_GENERATIONS   = 18       # fewer generations for GA
GA_POP           = 18

# ---- Paths ----
CANDIDATES = ["../data/raw/Dataset", "data/raw/Dataset"]
INPUT_DIR = next((p for p in CANDIDATES if os.path.isdir(p)), None)
if INPUT_DIR is None:
    raise FileNotFoundError("Put .XPT files in data/raw/Dataset")
OUTPUT_DIR = "../data/processed/mental_outputs"; Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# ---- Step 1: Domain-Aware Group Formation ----
DOMAIN_BY_MODULE: Dict[str, str] = {
    # Socio-demographic
    "DEMO":"socio-demographic","INQ":"socio-demographic","HIQ":"socio-demographic",
    "OCQ":"socio-demographic","FSQ":"socio-demographic",
    # Behavioral / Lifestyle
    "ALQ":"behavioral-lifestyle","SMQ":"behavioral-lifestyle","PAQ":"behavioral-lifestyle",
    "SLQ":"behavioral-lifestyle","DBQ":"behavioral-lifestyle","WHQ":"behavioral-lifestyle",
    # Clinical / Health
    "HSQ":"clinical-health","HUQ":"clinical-health","MCQ":"clinical-health","BPQ":"clinical-health",
    "OHQ":"clinical-health","RHQ":"clinical-health","HEQ":"clinical-health","RXQ":"clinical-health",
    "ACQ":"clinical-health","AUQ":"clinical-health",
    # Outcome
    "DPQ":"mental-health-outcome",
}
DPQ_VALID_VALUES = {0,1,2,3}

def infer_module_from_filename(stem: str) -> str:
    s = stem.upper();  return s[2:] if s.startswith("P_") else s

def coarse_module_key(module: str) -> str:
    m = module.upper()
    if m in DOMAIN_BY_MODULE: return m
    parts = m.split("_")
    for p in [parts[0], m[:3], m[:4]]:
        if p in DOMAIN_BY_MODULE: return p
    return parts[0] if parts else m

def assign_domain_from_module(module: str) -> str:
    return DOMAIN_BY_MODULE.get(coarse_module_key(module), "other")

# ---------- helpers: memory ----------
def downcast_numeric(df: pd.DataFrame) -> pd.DataFrame:
    for c in df.select_dtypes(include=["float64"]).columns:
        df[c] = pd.to_numeric(df[c], downcast="float")
    for c in df.select_dtypes(include=["int64","int32"]).columns:
        df[c] = pd.to_numeric(df[c], downcast="integer")
    return df

# ---------- person-level aggregation for long modules ----------
def _mode_agg(s: pd.Series):
    s = s.dropna()
    if s.empty: return np.nan
    m = s.mode()
    return m.iloc[0] if not m.empty else s.iloc[0]

def _coerce_numeric_if_mostly_numeric(s: pd.Series):
    if pd.api.types.is_object_dtype(s):
        sn = pd.to_numeric(s, errors="coerce")
        if sn.notna().mean() >= 0.8:
            return sn
    return s

def aggregate_module_person_level(df: pd.DataFrame, module: str) -> pd.DataFrame:
    d = df.copy(); d.columns = [str(c).upper() for c in d.columns]
    for c in d.columns:
        if c == "SEQN": continue
        d[c] = _coerce_numeric_if_mostly_numeric(d[c])
    rc = d.groupby("SEQN").size().rename(f"{module}_N").reset_index()
    agg_dict = {c: ("median" if pd.api.types.is_numeric_dtype(d[c]) else _mode_agg)
                for c in d.columns if c!="SEQN"}
    d_agg = d.groupby("SEQN", as_index=False).agg(agg_dict).merge(rc, on="SEQN", how="left")
    return downcast_numeric(d_agg)

# ---------- read & merge ----------
def read_all_xpt(input_dir: str) -> Dict[str, pd.DataFrame]:
    dfs = {}
    for p in sorted(Path(input_dir).glob("*.xpt")):
        module = infer_module_from_filename(p.stem)
        try:
            df = pd.read_sas(p, format="xport", encoding="utf-8")
        except Exception as e:
            print(f"[WARN] Can't read {p.name}: {e}"); continue
        df.columns = [str(c).upper() for c in df.columns]
        if "SEQN" not in df.columns:
            print(f"[INFO] {p.name} has no SEQN; skipped."); continue
        dfs[module] = downcast_numeric(df)
        dup = int(df.duplicated("SEQN").sum())
        print(f"[OK] {p.name:20s} -> {module:10s} shape={tuple(df.shape)}{' (long)' if dup>0 else ''}")
    return dfs

def merge_modules_person_level(mods: Dict[str,pd.DataFrame]) -> Tuple[pd.DataFrame, Dict[str,str]]:
    if not mods: raise RuntimeError("No modules.")
    start_key = "DEMO" if "DEMO" in mods else next(iter(mods.keys()))
    base = mods[start_key]
    if base.duplicated("SEQN").any():
        base = aggregate_module_person_level(base, start_key)
    merged = base.copy()
    col_domain = {c: assign_domain_from_module(start_key) for c in merged.columns if c!="SEQN"}

    for module, df in mods.items():
        if module == start_key: continue
        df2 = aggregate_module_person_level(df, module) if df.duplicated("SEQN").any() else downcast_numeric(df.copy())
        # avoid collisions
        for c in list(df2.columns):
            if c=="SEQN": continue
            if c in merged.columns:
                newc = f"{c}__{module}"
                df2.rename(columns={c:newc}, inplace=True)
                col_domain[newc] = assign_domain_from_module(module)
            else:
                col_domain[c] = assign_domain_from_module(module)
        merged = pd.merge(merged, df2, on="SEQN", how="outer", validate="one_to_one")
        del df2; gc.collect()
    return downcast_numeric(merged), col_domain

def build_phq9(df: pd.DataFrame) -> pd.DataFrame:
    dpq_cols = [c for c in df.columns if re.fullmatch(r"DPQ0[1-9]0(?:__.*)?", c)]
    if not dpq_cols:
        print("[WARN] No DPQ items found; PHQ-9 cannot be built.")
        out = df.copy(); out["PHQ9_TOTAL"]=np.nan; return out
    def _coerce(s):
        v = pd.to_numeric(s, errors="coerce")
        return v.where(v.isin(list(DPQ_VALID_VALUES)), np.nan)
    mat = pd.DataFrame({c: _coerce(df[c]) for c in dpq_cols})
    out = df.copy(); out["PHQ9_TOTAL"] = mat.sum(axis=1, min_count=1)
    return downcast_numeric(out)

# ----- merge & target -----
mods = read_all_xpt(INPUT_DIR)
merged, col_dom = merge_modules_person_level(mods)
del mods; gc.collect()
merged = build_phq9(merged)
json.dump(DOMAIN_BY_MODULE, open(f"{OUTPUT_DIR}/domain_mapping.json","w"), indent=2)
print("[SAVE] domain_mapping.json")

# ---- preprocessing ----
df = merged.copy(); del merged; gc.collect()
y = df["PHQ9_TOTAL"]; mask = y.notna()
df = df.loc[mask].copy(); y = y.loc[mask].astype("float32")

# optional row subsample for memory
if LOW_MEM and ROW_SUBSAMPLE and len(df) > ROW_SUBSAMPLE:
    df = df.sample(n=ROW_SUBSAMPLE, random_state=42)
    y  = y.loc[df.index]
    print(f"[INFO] Row subsample → {len(df)} rows")

drop_patterns = [r"^SEQN$", r"^PHQ9_TOTAL$", r"^DPQ"]
drop_cols = set()
for pat in drop_patterns: drop_cols |= set([c for c in df.columns if re.match(pat, c)])
X = df.drop(columns=list(drop_cols), errors="ignore"); del df; gc.collect()

# keep ≥60% observed
X = X.loc[:, X.notna().sum().ge(int(0.6*len(X)))].copy()

num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
cat_cols = [c for c in X.columns if c not in num_cols]

# OneHot with frequency cap to avoid explosion
try:
    ohe = OneHotEncoder(handle_unknown="ignore",
                        min_frequency=OHE_MIN_FREQ, max_categories=OHE_MAX_CATS,
                        sparse_output=False)
except TypeError:
    # older/newer API diff
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

prep = ColumnTransformer(
    [("num", SimpleImputer(strategy="median"), num_cols),
     ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", ohe)]), cat_cols)],
    remainder="drop", verbose_feature_names_out=False
)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
del X; gc.collect()

# ---- Step 2: Causal Feature Selection (SHAP → prune) ----
reg = HistGradientBoostingRegressor(loss="squared_error", learning_rate=0.08,
                                    max_leaf_nodes=31, random_state=42)
pipe = Pipeline([("prep", prep), ("model", reg)])
pipe.fit(Xtr, ytr)
yhat = pipe.predict(Xte)
print(f"[METRIC] Holdout R2={r2_score(yte,yhat):.3f} | MAE={mean_absolute_error(yte,yhat):.3f}")

feat_proc = pipe.named_steps["prep"].get_feature_names_out().tolist()
def base_name(f): return f.split("_")[0].split("=")[0]

# SHAP with row subsample (to avoid OOM); fallback to permutation
try:
    import shap
    Xte_proc_full = pipe.named_steps["prep"].transform(Xte)
    if LOW_MEM and SHAP_SAMPLE and Xte_proc_full.shape[0] > SHAP_SAMPLE:
        idx = np.random.RandomState(42).choice(Xte_proc_full.shape[0], SHAP_SAMPLE, replace=False)
        Xte_proc = Xte_proc_full[idx]
    else:
        Xte_proc = Xte_proc_full
    expl = shap.Explainer(pipe.named_steps["model"], feature_names=feat_proc, algorithm="tree")
    sv = expl(Xte_proc)
    shap_abs = np.abs(sv.values).mean(axis=0)
    imp_df = pd.DataFrame({"feature_proc": feat_proc, "shap_importance": shap_abs})
    imp_df["feature_base"] = imp_df["feature_proc"].map(base_name)
except Exception as e:
    print(f"[WARN] SHAP failed ({e}); using permutation importance (light).")
    res = permutation_importance(pipe, Xte, yte, n_repeats=PI_REPEATS, random_state=42, n_jobs=-1, scoring="r2")
    imp_df = pd.DataFrame({"feature_proc": feat_proc, "shap_importance": res.importances_mean})
    imp_df["feature_base"] = imp_df["feature_proc"].map(base_name)

agg = imp_df.groupby("feature_base", as_index=False)["shap_importance"].sum()
agg["domain"] = agg["feature_base"].map(lambda b: col_dom.get(b, "other"))
agg["group_rel"] = agg.groupby("domain")["shap_importance"].transform(lambda s: s/(s.sum()+1e-9))

PRUNE_REL_THRESHOLD = 0.02
TOPK_PER_GROUP = 10
keep = (agg.query("group_rel >= @PRUNE_REL_THRESHOLD")
          .sort_values(["domain","shap_importance"], ascending=[True,False])
          .groupby("domain", as_index=False).head(TOPK_PER_GROUP))
keep.to_csv(f"{OUTPUT_DIR}/groupwise_pruned_features.csv", index=False)
agg.sort_values(["domain","shap_importance"], ascending=[True,False]) \
   .to_csv(f"{OUTPUT_DIR}/groupwise_ranked_features_full.csv", index=False)
print("[SAVE] groupwise_pruned_features.csv")
print("[SAVE] groupwise_ranked_features_full.csv")

# ---- Step 3: Domain Prioritization (EBM if available, else GAM) ----
try:
    from interpret.glassbox import ExplainableBoostingRegressor
    EBM_AVAILABLE = True
except Exception:
    from pygam import LinearGAM
    EBM_AVAILABLE = False
import pygad

def cols_for_domains(cols: List[str], domains_selected: List[str]) -> List[str]:
    return [c for c in cols if col_dom.get(c, "other") in domains_selected]

def eval_domains(domains_selected: List[str]):
    base_cols = cols_for_domains(list(Xtr.columns) if isinstance(Xtr, pd.DataFrame) else list(pipe.named_steps["prep"].feature_names_in_), domains_selected)
    # rebuild Xsub from original columns present in train/val split
    Xsub_all = pd.concat([Xtr, Xte], axis=0) if isinstance(Xtr, pd.DataFrame) else None
    if Xsub_all is None:  # fallback when ColumnTransformer dropped pandas index
        return {"r2": -np.inf, "mae": np.inf, "n_features": 0}
    Xsub = Xsub_all[base_cols].copy()
    y_all = pd.concat([pd.Series(ytr), pd.Series(yte)], axis=0)

    num = [c for c in Xsub.columns if pd.api.types.is_numeric_dtype(Xsub[c])]
    cat = [c for c in Xsub.columns if c not in num]
    try:
        ohe2 = OneHotEncoder(handle_unknown="ignore",
                             min_frequency=OHE_MIN_FREQ, max_categories=OHE_MAX_CATS,
                             sparse_output=False)
    except TypeError:
        ohe2 = OneHotEncoder(handle_unknown="ignore", sparse=False)
    prep2 = ColumnTransformer(
        [("num", SimpleImputer(strategy="median"), num),
         ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", ohe2)]), cat)],
        remainder="drop", verbose_feature_names_out=False
    )
    Xtr2, Xva2, ytr2, yva2 = train_test_split(Xsub, y_all, test_size=0.25, random_state=12)

    if EBM_AVAILABLE:
        est = ExplainableBoostingRegressor(random_state=12)
        pipe2 = Pipeline([("prep", prep2), ("model", est)])
        pipe2.fit(Xtr2, ytr2)
        yhat2 = pipe2.predict(Xva2)
        r2 = r2_score(yva2, yhat2); mae = mean_absolute_error(yva2, yhat2)
        n_feat = pipe2.named_steps["prep"].get_feature_names_out().size
        return {"r2": r2, "mae": mae, "n_features": int(n_feat), "model": pipe2, "is_ebm": True}
    else:
        Xtr2p, Xva2p = prep2.fit_transform(Xtr2), prep2.transform(Xva2)
        gam = LinearGAM().gridsearch(Xtr2p, ytr2)
        yhat2 = gam.predict(Xva2p)
        r2 = r2_score(yva2, yhat2); mae = mean_absolute_error(yva2, yhat2)
        n_feat = Xtr2p.shape[1]
        class Dummy:
            def __init__(self, prep, model): self.named_steps={"prep":prep, "model":model}
        return {"r2": r2, "mae": mae, "n_features": int(n_feat), "model": Dummy(prep2, gam), "is_ebm": False}

# 3A) Domain contribution in processed space
all_domains = sorted(set(col_dom.get(c,"other") for c in keep["feature_base"])) \
              or sorted(set(col_dom.get(c,"other") for c in agg["feature_base"]))
pipe_dom = eval_domains(all_domains)["model"]
prep2 = pipe_dom.named_steps["prep"]; est = pipe_dom.named_steps["model"]
# use validation part (Xte) columns
X_eval_base = Xte[[c for c in Xte.columns if col_dom.get(c, "other") in all_domains]].copy()
X_eval_proc = prep2.transform(X_eval_base)
feat_names_proc = prep2.get_feature_names_out().tolist()
pi = permutation_importance(est, X_eval_proc, yte, n_repeats=PI_REPEATS, random_state=7, n_jobs=-1, scoring="r2")
assert X_eval_proc.shape[1] == len(feat_names_proc) == len(pi.importances_mean)

dom_of_proc = {fn: col_dom.get(fn.split("_")[0].split("=")[0], "other") for fn in feat_names_proc}
dom_imp = pd.DataFrame({"feature_proc": feat_names_proc, "perm_importance": pi.importances_mean})
dom_imp["domain"] = dom_imp["feature_proc"].map(lambda f: dom_of_proc.get(f, "other"))
dom_summary = dom_imp.groupby("domain", as_index=False)["perm_importance"].sum() \
                     .sort_values("perm_importance", ascending=False)
dom_summary.to_csv(f"{OUTPUT_DIR}/domain_importance_ebm.csv", index=False)
print("[SAVE] domain_importance_ebm.csv")

# 3B) GA (PyGAD ≥2.20: 3-arg fitness)
import pygad
domain_list = all_domains; D = len(domain_list)
ALPHA, BETA, GAMMA = 1.0, 0.0005, 0.01

def fitness_func(ga_instance, solution, solution_idx):
    bits = [int(round(b)) for b in solution]
    selected = [domain_list[i] for i,b in enumerate(bits) if b==1]
    if not selected: return -1e9
    res = eval_domains(selected)
    if not np.isfinite(res["r2"]): return -1e9
    return float((ALPHA*res["r2"]) - (BETA*res["n_features"]) - (GAMMA*len(selected)))

initial_pop = np.random.randint(0,2,size=(GA_POP, D))
ga = pygad.GA(num_generations=GA_GENERATIONS, num_parents_mating=6,
              fitness_func=fitness_func, sol_per_pop=GA_POP, num_genes=D,
              gene_space=[0,1], gene_type=int, initial_population=initial_pop,
              mutation_probability=0.12, allow_duplicate_genes=True, suppress_warnings=True)
ga.run()
best_sol, best_fitness, _ = ga.best_solution()
best_bits = [int(round(b)) for b in best_sol]
best_domains = [domain_list[i] for i,b in enumerate(best_bits) if b==1]
res_best = eval_domains(best_domains)

ga_out = {
    "domain_list": domain_list,
    "best_domains": best_domains,
    "fitness": float(best_fitness),
    "r2": float(res_best["r2"]), "mae": float(res_best["mae"]),
    "n_features": int(res_best["n_features"])
}
json.dump(ga_out, open(f"{OUTPUT_DIR}/ga_domain_selection.json","w"), indent=2)
print("[SAVE] ga_domain_selection.json")

dom_summary["selected_by_GA"] = dom_summary["domain"].isin(best_domains).astype(int)
dom_summary.to_csv(f"{OUTPUT_DIR}/domain_priority_final.csv", index=False)
print("[SAVE] domain_priority_final.csv")

from IPython.display import display
print("\n--- Domain importance (perm) ---"); display(dom_summary)
print("\n--- GA selected domains ---"); print(best_domains)

# quick viz (light)
plt.figure(); top = keep.sort_values("shap_importance", ascending=False).head(20)
plt.barh((top["feature_base"]+" ["+top["domain"]+"]")[::-1], top["shap_importance"][::-1])
plt.title("Top 20 Pruned Features (Causal score)"); plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR)/"viz_top_pruned_features.png", dpi=200); plt.show()

plt.figure(); ds = dom_summary.sort_values("perm_importance", ascending=False)
lbl = ds["domain"] + ds["selected_by_GA"].map({1:" ★",0:""})
plt.barh(lbl[::-1], ds["perm_importance"][::-1])
plt.title("Domain Importance & GA selection"); plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR)/"viz_domain_priority_final.png", dpi=200); plt.show()


Full update after requirements

In [ ]:
# ============================================
# Domain-First Causal Pipeline (Steps 1–3) — Py3.12 + Low-Memory safe
# + 3C) Validation & Comparative Evaluation (Ours vs. SGFR)  [LOW-MEM EDITION]
# ============================================
import os, re, json, gc, warnings, itertools
from pathlib import Path
from typing import Dict, List, Tuple
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=FutureWarning)

# ---------- Low-memory toggles ----------
LOW_MEM           = True
ROW_SUBSAMPLE     = 30000    # max rows for modeling (None to disable)
SHAP_SAMPLE       = 2000     # rows for SHAP explain (smaller -> lighter)
PI_REPEATS        = 2        # permutation repeats (lighter)
OHE_MIN_FREQ      = 0.02     # collapse rarer categories
OHE_MAX_CATS      = 25       # cap category explosion
GA_GENERATIONS    = 8        # fewer generations for GA
GA_POP            = 12
FORCE_GAM         = True     # << use LinearGAM instead of EBM to avoid heavy training
EVAL_SAMPLE       = 2000     # rows for permutation-importance in Step 3A (processed space)

# ---- Classification toggle for SGFR comparison ----
BINARY_OUTCOME        = True            # set False to skip classification panel
PHQ9_BINARY_THRESHOLD = 10              # PHQ-9 >= 10 => 1 else 0

# ---- Paths ----
CANDIDATES = ["../data/raw/Dataset", "data/raw/Dataset"]
INPUT_DIR = next((p for p in CANDIDATES if os.path.isdir(p)), None)
if INPUT_DIR is None:
    raise FileNotFoundError("Put .XPT files in data/raw/Dataset")
OUTPUT_DIR = "../data/processed/mental_outputs"; Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# ---- Step 1: Domain-Aware Group Formation ----
DOMAIN_BY_MODULE: Dict[str, str] = {
    # Socio-demographic
    "DEMO":"socio-demographic","INQ":"socio-demographic","HIQ":"socio-demographic",
    "OCQ":"socio-demographic","FSQ":"socio-demographic",
    # Behavioral / Lifestyle
    "ALQ":"behavioral-lifestyle","SMQ":"behavioral-lifestyle","PAQ":"behavioral-lifestyle",
    "SLQ":"behavioral-lifestyle","DBQ":"behavioral-lifestyle","WHQ":"behavioral-lifestyle",
    # Clinical / Health
    "HSQ":"clinical-health","HUQ":"clinical-health","MCQ":"clinical-health","BPQ":"clinical-health",
    "OHQ":"clinical-health","RHQ":"clinical-health","HEQ":"clinical-health","RXQ":"clinical-health",
    "ACQ":"clinical-health","AUQ":"clinical-health",
    # Outcome
    "DPQ":"mental-health-outcome",
}
DPQ_VALID_VALUES = {0,1,2,3}

def infer_module_from_filename(stem: str) -> str:
    s = stem.upper();  return s[2:] if s.startswith("P_") else s

def coarse_module_key(module: str) -> str:
    m = module.upper()
    if m in DOMAIN_BY_MODULE: return m
    parts = m.split("_")
    for p in [parts[0], m[:3], m[:4]]:
        if p in DOMAIN_BY_MODULE: return p
    return parts[0] if parts else m

def assign_domain_from_module(module: str) -> str:
    return DOMAIN_BY_MODULE.get(coarse_module_key(module), "other")

# ---------- helpers: memory ----------
def downcast_numeric(df: pd.DataFrame) -> pd.DataFrame:
    for c in df.select_dtypes(include=["float64"]).columns:
        df[c] = pd.to_numeric(df[c], downcast="float")
    for c in df.select_dtypes(include=["int64","int32"]).columns:
        df[c] = pd.to_numeric(df[c], downcast="integer")
    return df

# ---------- person-level aggregation for long modules ----------
def _mode_agg(s: pd.Series):
    s = s.dropna()
    if s.empty: return np.nan
    m = s.mode()
    return m.iloc[0] if not m.empty else s.iloc[0]

def _coerce_numeric_if_mostly_numeric(s: pd.Series):
    if pd.api.types.is_object_dtype(s):
        sn = pd.to_numeric(s, errors="coerce")
        if sn.notna().mean() >= 0.8:
            return sn
    return s

def aggregate_module_person_level(df: pd.DataFrame, module: str) -> pd.DataFrame:
    d = df.copy(); d.columns = [str(c).upper() for c in d.columns]
    for c in d.columns:
        if c == "SEQN": continue
        d[c] = _coerce_numeric_if_mostly_numeric(d[c])
    rc = d.groupby("SEQN").size().rename(f"{module}_N").reset_index()
    agg_dict = {c: ("median" if pd.api.types.is_numeric_dtype(d[c]) else _mode_agg)
                for c in d.columns if c!="SEQN"}
    d_agg = d.groupby("SEQN", as_index=False).agg(agg_dict).merge(rc, on="SEQN", how="left")
    return downcast_numeric(d_agg)

# ---------- read & merge ----------
def read_all_xpt(input_dir: str) -> Dict[str, pd.DataFrame]:
    dfs = {}
    for p in sorted(Path(input_dir).glob("*.xpt")):
        module = infer_module_from_filename(p.stem)
        try:
            df = pd.read_sas(p, format="xport", encoding="utf-8")
        except Exception as e:
            print(f"[WARN] Can't read {p.name}: {e}"); continue
        df.columns = [str(c).upper() for c in df.columns]
        if "SEQN" not in df.columns:
            print(f"[INFO] {p.name} has no SEQN; skipped."); continue
        dfs[module] = downcast_numeric(df)
        dup = int(df.duplicated("SEQN").sum())
        print(f"[OK] {p.name:20s} -> {module:10s} shape={tuple(df.shape)}{' (long)' if dup>0 else ''}")
    return dfs

def merge_modules_person_level(mods: Dict[str,pd.DataFrame]) -> Tuple[pd.DataFrame, Dict[str,str]]:
    if not mods: raise RuntimeError("No modules.")
    start_key = "DEMO" if "DEMO" in mods else next(iter(mods.keys()))
    base = mods[start_key]
    if base.duplicated("SEQN").any():
        base = aggregate_module_person_level(base, start_key)
    merged = base.copy()
    col_domain = {c: assign_domain_from_module(start_key) for c in merged.columns if c!="SEQN"}
    for module, df in mods.items():
        if module == start_key: continue
        df2 = aggregate_module_person_level(df, module) if df.duplicated("SEQN").any() else downcast_numeric(df.copy())
        for c in list(df2.columns):
            if c=="SEQN": continue
            if c in merged.columns:
                newc = f"{c}__{module}"
                df2.rename(columns={c:newc}, inplace=True)
                col_domain[newc] = assign_domain_from_module(module)
            else:
                col_domain[c] = assign_domain_from_module(module)
        merged = pd.merge(merged, df2, on="SEQN", how="outer", validate="one_to_one")
        del df2; gc.collect()
    return downcast_numeric(merged), col_domain

def build_phq9(df: pd.DataFrame) -> pd.DataFrame:
    dpq_cols = [c for c in df.columns if re.fullmatch(r"DPQ0[1-9]0(?:__.*)?", c)]
    if not dpq_cols:
        print("[WARN] No DPQ items found; PHQ-9 cannot be built.")
        out = df.copy(); out["PHQ9_TOTAL"]=np.nan; return out
    def _coerce(s):
        v = pd.to_numeric(s, errors="coerce")
        return v.where(v.isin(list(DPQ_VALID_VALUES)), np.nan)
    mat = pd.DataFrame({c: _coerce(df[c]) for c in dpq_cols})
    out = df.copy(); out["PHQ9_TOTAL"] = mat.sum(axis=1, min_count=1)
    return downcast_numeric(out)

# ----- merge & target -----
mods = read_all_xpt(INPUT_DIR)
merged, col_dom = merge_modules_person_level(mods)
del mods; gc.collect()
merged = build_phq9(merged)
json.dump(DOMAIN_BY_MODULE, open(f"{OUTPUT_DIR}/domain_mapping.json","w"), indent=2)
print("[SAVE] domain_mapping.json")

# ---- preprocessing ----
df = merged.copy(); del merged; gc.collect()
y = df["PHQ9_TOTAL"]; mask = y.notna()
df = df.loc[mask].copy(); y = y.loc[mask].astype("float32")

# optional row subsample for memory
if LOW_MEM and ROW_SUBSAMPLE and len(df) > ROW_SUBSAMPLE:
    df = df.sample(n=ROW_SUBSAMPLE, random_state=42)
    y  = y.loc[df.index]
    print(f"[INFO] Row subsample → {len(df)} rows")

drop_patterns = [r"^SEQN$", r"^PHQ9_TOTAL$", r"^DPQ"]
drop_cols = set()
for pat in drop_patterns: drop_cols |= set([c for c in df.columns if re.match(pat, c)])
X = df.drop(columns=list(drop_cols), errors="ignore"); del df; gc.collect()

# keep ≥60% observed
X = X.loc[:, X.notna().sum().ge(int(0.6*len(X)))].copy()

num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
cat_cols = [c for c in X.columns if c not in num_cols]

# OneHot with frequency cap to avoid explosion
try:
    ohe = OneHotEncoder(handle_unknown="ignore",
                        min_frequency=OHE_MIN_FREQ, max_categories=OHE_MAX_CATS,
                        sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

prep = ColumnTransformer(
    [("num", SimpleImputer(strategy="median"), num_cols),
     ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", ohe)]), cat_cols)],
    remainder="drop", verbose_feature_names_out=False
)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
del X; gc.collect()

# ---- Step 2: Causal Feature Selection (SHAP → prune) ----
reg = HistGradientBoostingRegressor(loss="squared_error", learning_rate=0.08,
                                    max_leaf_nodes=31, random_state=42)
pipe = Pipeline([("prep", prep), ("model", reg)])
pipe.fit(Xtr, ytr)
yhat = pipe.predict(Xte)
print(f"[METRIC] Holdout R2={r2_score(yte,yhat):.3f} | MAE={mean_absolute_error(yte,yhat):.3f}")

feat_proc = pipe.named_steps["prep"].get_feature_names_out().tolist()
def base_name(f): return f.split("_")[0].split("=")[0]

# SHAP with row subsample (to avoid OOM); fallback to permutation
try:
    import shap
    Xte_proc_full = pipe.named_steps["prep"].transform(Xte)
    if LOW_MEM and SHAP_SAMPLE and Xte_proc_full.shape[0] > SHAP_SAMPLE:
        idx = np.random.RandomState(42).choice(Xte_proc_full.shape[0], SHAP_SAMPLE, replace=False)
        Xte_proc = Xte_proc_full[idx]
    else:
        Xte_proc = Xte_proc_full
    expl = shap.Explainer(pipe.named_steps["model"], feature_names=feat_proc, algorithm="tree")
    sv = expl(Xte_proc)
    shap_abs = np.abs(sv.values).mean(axis=0)
    imp_df = pd.DataFrame({"feature_proc": feat_proc, "shap_importance": shap_abs})
    imp_df["feature_base"] = imp_df["feature_proc"].map(base_name)
except Exception as e:
    print(f"[WARN] SHAP failed ({e}); using permutation importance (light).")
    res = permutation_importance(pipe, Xte, yte, n_repeats=PI_REPEATS, random_state=42, n_jobs=-1, scoring="r2")
    imp_df = pd.DataFrame({"feature_proc": feat_proc, "shap_importance": res.importances_mean})
    imp_df["feature_base"] = imp_df["feature_proc"].map(base_name)

agg = imp_df.groupby("feature_base", as_index=False)["shap_importance"].sum()
agg["domain"] = agg["feature_base"].map(lambda b: col_dom.get(b, "other"))
agg["group_rel"] = agg.groupby("domain")["shap_importance"].transform(lambda s: s/(s.sum()+1e-9))

PRUNE_REL_THRESHOLD = 0.02
TOPK_PER_GROUP = 10
keep = (agg.query("group_rel >= @PRUNE_REL_THRESHOLD")
          .sort_values(["domain","shap_importance"], ascending=[True,False])
          .groupby("domain", as_index=False).head(TOPK_PER_GROUP))
keep.to_csv(f"{OUTPUT_DIR}/groupwise_pruned_features.csv", index=False)
agg.sort_values(["domain","shap_importance"], ascending=[True,False]) \
   .to_csv(f"{OUTPUT_DIR}/groupwise_ranked_features_full.csv", index=False)
print("[SAVE] groupwise_pruned_features.csv")
print("[SAVE] groupwise_ranked_features_full.csv")

# ---- Step 3: Domain Prioritization (LinearGAM forced by default) ----
if FORCE_GAM:
    from pygam import LinearGAM
    EBM_AVAILABLE = False
else:
    try:
        from interpret.glassbox import ExplainableBoostingRegressor
        EBM_AVAILABLE = True
    except Exception:
        from pygam import LinearGAM
        EBM_AVAILABLE = False

import pygad

def eval_domains(domains_selected: List[str]):
    # << LOW-MEM: restrict to pruned bases only >>
    kept_bases = set(keep["feature_base"].tolist())
    candidate_cols = list(Xtr.columns) if isinstance(Xtr, pd.DataFrame) \
                     else list(pipe.named_steps["prep"].feature_names_in_)
    base_cols = [c for c in candidate_cols
                 if (col_dom.get(c, "other") in domains_selected) and (c in kept_bases)]
    if not base_cols:
        return {"r2": -np.inf, "mae": np.inf, "n_features": 0}

    Xsub_all = pd.concat([Xtr, Xte], axis=0) if isinstance(Xtr, pd.DataFrame) else None
    if Xsub_all is None:
        return {"r2": -np.inf, "mae": np.inf, "n_features": 0}
    Xsub = Xsub_all[base_cols].copy()
    y_all = pd.concat([pd.Series(ytr), pd.Series(yte)], axis=0)

    num = [c for c in Xsub.columns if pd.api.types.is_numeric_dtype(Xsub[c])]
    cat = [c for c in Xsub.columns if c not in num]
    try:
        ohe2 = OneHotEncoder(handle_unknown="ignore",
                             min_frequency=OHE_MIN_FREQ, max_categories=OHE_MAX_CATS,
                             sparse_output=False)
    except TypeError:
        ohe2 = OneHotEncoder(handle_unknown="ignore", sparse=False)
    prep2 = ColumnTransformer(
        [("num", SimpleImputer(strategy="median"), num),
         ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", ohe2)]), cat)],
        remainder="drop", verbose_feature_names_out=False
    )
    Xtr2, Xva2, ytr2, yva2 = train_test_split(Xsub, y_all, test_size=0.25, random_state=12)

    if EBM_AVAILABLE:
        est = ExplainableBoostingRegressor(
            random_state=12, max_bins=128, interactions=0,
            outer_bags=4, inner_bags=0, learning_rate=0.02, n_jobs=-1
        )
        pipe2 = Pipeline([("prep", prep2), ("model", est)])
        pipe2.fit(Xtr2, ytr2)
        yhat2 = pipe2.predict(Xva2)
        r2 = r2_score(yva2, yhat2); mae = mean_absolute_error(yva2, yhat2)
        n_feat = pipe2.named_steps["prep"].get_feature_names_out().size
        return {"r2": r2, "mae": mae, "n_features": int(n_feat), "model": pipe2, "is_ebm": True}
    else:
        Xtr2p, Xva2p = prep2.fit_transform(Xtr2), prep2.transform(Xva2)
        gam = LinearGAM().gridsearch(Xtr2p, ytr2)
        yhat2 = gam.predict(Xva2p)
        r2 = r2_score(yva2, yhat2); mae = mean_absolute_error(yva2, yhat2)
        n_feat = Xtr2p.shape[1]
        class Dummy:
            def __init__(self, prep, model): self.named_steps={"prep":prep, "model":model}
        return {"r2": r2, "mae": mae, "n_features": int(n_feat), "model": Dummy(prep2, gam), "is_ebm": False}

# 3A) Domain contribution in processed space (LOW-MEM sampling)
all_domains = sorted(set(col_dom.get(c,"other") for c in keep["feature_base"])) \
              or sorted(set(col_dom.get(c,"other") for c in agg["feature_base"]))
pipe_dom = eval_domains(all_domains)["model"]
prep2 = pipe_dom.named_steps["prep"]; est = pipe_dom.named_steps["model"]

X_eval_base = Xte[[c for c in Xte.columns if col_dom.get(c, "other") in all_domains]].copy()
X_eval_proc_full = prep2.transform(X_eval_base)
if LOW_MEM and EVAL_SAMPLE and X_eval_proc_full.shape[0] > EVAL_SAMPLE:
    ridx = np.random.RandomState(7).choice(X_eval_proc_full.shape[0], EVAL_SAMPLE, replace=False)
    X_eval_proc = X_eval_proc_full[ridx]
    y_eval = yte.iloc[ridx]
else:
    X_eval_proc = X_eval_proc_full
    y_eval = yte

feat_names_proc = prep2.get_feature_names_out().tolist()
pi = permutation_importance(est, X_eval_proc, y_eval, n_repeats=PI_REPEATS, random_state=7, n_jobs=-1, scoring="r2")
assert X_eval_proc.shape[1] == len(feat_names_proc) == len(pi.importances_mean)

dom_of_proc = {fn: col_dom.get(fn.split("_")[0].split("=")[0], "other") for fn in feat_names_proc}
dom_imp = pd.DataFrame({"feature_proc": feat_names_proc, "perm_importance": pi.importances_mean})
dom_imp["domain"] = dom_imp["feature_proc"].map(lambda f: dom_of_proc.get(f, "other"))
dom_summary = dom_imp.groupby("domain", as_index=False)["perm_importance"].sum() \
                     .sort_values("perm_importance", ascending=False)
dom_summary.to_csv(f"{OUTPUT_DIR}/domain_importance_ebm.csv", index=False)
print("[SAVE] domain_importance_ebm.csv")

# 3B) GA (PyGAD ≥2.20: 3-arg fitness) — light config
import pygad
domain_list = all_domains; D = len(domain_list)
ALPHA, BETA, GAMMA = 1.0, 0.0005, 0.01

def fitness_func(ga_instance, solution, solution_idx):
    bits = [int(round(b)) for b in solution]
    selected = [domain_list[i] for i,b in enumerate(bits) if b==1]
    if not selected: return -1e9
    res = eval_domains(selected)
    if not np.isfinite(res["r2"]): return -1e9
    return float((ALPHA*res["r2"]) - (BETA*res["n_features"]) - (GAMMA*len(selected)))

initial_pop = np.random.randint(0,2,size=(GA_POP, D))
ga = pygad.GA(num_generations=GA_GENERATIONS, num_parents_mating=6,
              fitness_func=fitness_func, sol_per_pop=GA_POP, num_genes=D,
              gene_space=[0,1], gene_type=int, initial_population=initial_pop,
              mutation_probability=0.12, allow_duplicate_genes=True, suppress_warnings=True)
ga.run()
best_sol, best_fitness, _ = ga.best_solution()
best_bits = [int(round(b)) for b in best_sol]
best_domains = [domain_list[i] for i,b in enumerate(best_bits) if b==1]
res_best = eval_domains(best_domains)

ga_out = {
    "domain_list": domain_list,
    "best_domains": best_domains,
    "fitness": float(best_fitness),
    "r2": float(res_best["r2"]), "mae": float(res_best["mae"]),
    "n_features": int(res_best["n_features"])
}
json.dump(ga_out, open(f"{OUTPUT_DIR}/ga_domain_selection.json","w"), indent=2)
print("[SAVE] ga_domain_selection.json")

dom_summary["selected_by_GA"] = dom_summary["domain"].isin(best_domains).astype(int)
dom_summary.to_csv(f"{OUTPUT_DIR}/domain_priority_final.csv", index=False)
print("[SAVE] domain_priority_final.csv")

try:
    from IPython.display import display
    print("\n--- Domain importance (perm) ---"); display(dom_summary)
    print("\n--- GA selected domains ---"); print(best_domains)
except Exception:
    pass

# quick viz (light)
plt.figure(); top = keep.sort_values("shap_importance", ascending=False).head(20)
plt.barh((top["feature_base"]+" ["+top["domain"]+"]")[::-1], top["shap_importance"][::-1])
plt.title("Top 20 Pruned Features (Causal score)"); plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR)/"viz_top_pruned_features.png", dpi=200); plt.close()

plt.figure(); ds = dom_summary.sort_values("perm_importance", ascending=False)
lbl = ds["domain"] + ds["selected_by_GA"].map({1:" ★",0:""})
plt.barh(lbl[::-1], ds["perm_importance"][::-1])
plt.title("Domain Importance & GA selection"); plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR)/"viz_domain_priority_final.png", dpi=200); plt.close()

# ============================================================
# 3C) Validation & Comparative Evaluation (Ours vs. SGFR)
#     (Place BEFORE Step-4 Counterfactuals)
# ============================================================
if BINARY_OUTCOME:
    ytr_bin = (ytr >= PHQ9_BINARY_THRESHOLD).astype(int)
    yte_bin = (yte >= PHQ9_BINARY_THRESHOLD).astype(int)

    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import SVC
    from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
    from sklearn.calibration import CalibratedClassifierCV
    from sklearn.metrics import roc_auc_score, f1_score, balanced_accuracy_score, brier_score_loss, roc_curve
    from sklearn.preprocessing import KBinsDiscretizer
    from scipy.stats import wilcoxon

    def _select_cols_from_bases(df_cols, base_names):
        base_names = set(base_names)
        return [c for c in df_cols if c in base_names]

    def bases_from_keep(keep_df, domain_subset=None):
        if domain_subset is None:
            return keep_df["feature_base"].tolist()
        return keep_df.loc[keep_df["domain"].isin(domain_subset), "feature_base"].tolist()

    our_base_cols = bases_from_keep(keep, domain_subset=None)

    USE_OFFICIAL_SGFR = False
    SGFR_RANK_PATH = "data/processed/SGFR_rank.csv"
    if USE_OFFICIAL_SGFR and os.path.exists(SGFR_RANK_PATH):
        sgfr_rank = pd.read_csv(SGFR_RANK_PATH)
        sgfr_base_cols = sgfr_rank["feature_base"].tolist()
    else:
        def sgfr_proxy_rank(X_df, domains_map, topk_per_domain=10):
            out = []
            for g in sorted(set(domains_map.get(c, "other") for c in X_df.columns)):
                cols_g = [c for c in X_df.columns if domains_map.get(c, "other")==g]
                if len(cols_g) < 2:
                    out += cols_g;
                    continue
                Xg = X_df[cols_g].copy()
                Xg = Xg.fillna(Xg.median(numeric_only=True))
                from sklearn.decomposition import PCA
                num_cols_g = [c for c in cols_g if np.issubdtype(Xg[c].dtype, np.number)]
                if len(num_cols_g) < 2:
                    out += cols_g[:min(topk_per_domain, len(cols_g))]
                    continue
                pca = PCA(n_components=1, random_state=7).fit(Xg[num_cols_g])
                load = np.abs(pca.components_[0])
                order = np.argsort(load)[::-1]
                pick = [num_cols_g[i] for i in order[:min(topk_per_domain, len(order))]]
                out += pick
            return list(dict.fromkeys(out))
        sgfr_base_cols = sgfr_proxy_rank(Xtr, col_dom, topk_per_domain=TOPK_PER_GROUP)

    def build_pipe_for_cols(base_cols):
        base_cols = _select_cols_from_bases(list(Xtr.columns), base_cols)
        if not base_cols:
            raise RuntimeError("No columns selected for the given feature set.")
        num = [c for c in base_cols if pd.api.types.is_numeric_dtype(Xtr[c])]
        cat = [c for c in base_cols if c not in num]
        try:
            ohe2 = OneHotEncoder(handle_unknown="ignore",
                                 min_frequency=OHE_MIN_FREQ, max_categories=OHE_MAX_CATS,
                                 sparse_output=False)
        except TypeError:
            ohe2 = OneHotEncoder(handle_unknown="ignore", sparse=False)
        prep2 = ColumnTransformer(
            [("num", SimpleImputer(strategy="median"), num),
             ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", ohe2)]), cat)],
            remainder="drop", verbose_feature_names_out=False
        )
        return prep2, base_cols

    prep_ours, cols_o = build_pipe_for_cols(our_base_cols)
    prep_sgfr, cols_s = build_pipe_for_cols(sgfr_base_cols)

    Xtr_o = prep_ours.fit_transform(Xtr[cols_o]); Xte_o = prep_ours.transform(Xte[cols_o])
    Xtr_s = prep_sgfr.fit_transform(Xtr[cols_s]); Xte_s = prep_sgfr.transform(Xte[cols_s])

    clf_specs = {
        "LR":  LogisticRegression(max_iter=2000),
        "SVM": SVC(probability=True),
        "RF":  RandomForestClassifier(n_estimators=400, random_state=42),
        "GBC": GradientBoostingClassifier(random_state=42),
    }

    def eval_panel(Xtr_, ytr_, Xte_, yte_, tag):
        rows = []
        roc_curves = {}
        for name, base in clf_specs.items():
            cal = CalibratedClassifierCV(base, method="isotonic", cv=5)
            cal.fit(Xtr_, ytr_)
            prob = cal.predict_proba(Xte_)[:,1]
            pred = (prob >= 0.5).astype(int)
            auc  = roc_auc_score(yte_, prob)
            f1   = f1_score(yte_, pred, average="macro")
            bacc = balanced_accuracy_score(yte_, pred)
            brier = brier_score_loss(yte_, prob)
            fpr, tpr, _ = roc_curve(yte_, prob)
            roc_curves[name] = (fpr, tpr)
            rows.append({"Model": name, "Tag": tag, "AUC": auc, "MacroF1": f1, "BalancedAcc": bacc, "Brier": brier})
        return pd.DataFrame(rows), roc_curves

    res_ours, roc_ours = eval_panel(Xtr_o, ytr_bin, Xte_o, yte_bin, tag="Ours")
    res_sgfr, roc_sgfr = eval_panel(Xtr_s, ytr_bin, Xte_s, yte_bin, tag="SGFR")

    results_cmp = pd.concat([res_ours, res_sgfr], axis=0).reset_index(drop=True)
    results_cmp.to_csv(Path(OUTPUT_DIR)/"cmp_panel_metrics.csv", index=False)
    print("[SAVE] cmp_panel_metrics.csv")

    def safe_subgroups(df_base):
        subs = {}
        if "RIAGENDR" in df_base.columns:  # sex
            subs["sex"] = df_base["RIAGENDR"].map({1:"Male",2:"Female"})
        if "RIDAGEYR" in df_base.columns:
            bins = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="quantile")
            subs["age3"] = pd.Series(bins.fit_transform(df_base[["RIDAGEYR"]]).astype(int).ravel(), index=df_base.index)
        if "RIDRETH1" in df_base.columns:  # race/eth
            subs["race"] = df_base["RIDRETH1"]
        return subs

    def subgroup_auc(Xtr_, ytr_, Xte_, yte_, base_df_test, tag):
        subs = safe_subgroups(base_df_test)
        rows = []
        if not subs:
            return pd.DataFrame()
        for name, base in clf_specs.items():
            cal = CalibratedClassifierCV(base, method="isotonic", cv=3).fit(Xtr_, ytr_)
            prob = cal.predict_proba(Xte_)[:,1]
            for gname, gseries in subs.items():
                for gval in pd.Series(gseries).dropna().unique():
                    idx = (gseries==gval).values
                    if idx.sum() < 30:
                        continue
                    auc = roc_auc_score(yte_[idx], prob[idx])
                    rows.append({"Model": name, "Tag": tag, "Group": gname, "Level": str(gval), "AUC": auc})
        return pd.DataFrame(rows)

    sg_ours = subgroup_auc(Xtr_o, ytr_bin, Xte_o, yte_bin, Xte, "Ours")
    sg_sgfr = subgroup_auc(Xtr_s, ytr_bin, Xte_s, yte_bin, Xte, "SGFR")
    pd.concat([sg_ours, sg_sgfr]).to_csv(Path(OUTPUT_DIR)/"cmp_subgroup_auc.csv", index=False)
    print("[SAVE] cmp_subgroup_auc.csv")

    def wilcoxon_table(metric="AUC"):
        methods = ["Ours", "SGFR"]
        models = sorted(results_cmp["Model"].unique())
        perf = {
            "Ours": results_cmp.query("Tag=='Ours'").set_index("Model")[metric],
            "SGFR": results_cmp.query("Tag=='SGFR'").set_index("Model")[metric],
        }
        rank_mat = pd.DataFrame(0.0, index=methods, columns=methods)
        rej_map  = pd.DataFrame("", index=methods, columns=methods)
        for r, c in itertools.permutations(methods, 2):
            d = perf[r].loc[models].values - perf[c].loc[models].values
            try:
                stat, p = wilcoxon(d, zero_method="wilcox", alternative="two-sided", correction=True, mode="auto")
            except ValueError:
                stat, p = np.nan, 1.0
            pos_rank = float((d > 0).sum())
            rank_mat.loc[r, c] = pos_rank
            rej_map.loc[r, c]  = ("blue" if (p<0.05 and d.mean()>0) else ("red" if p<0.05 else ""))
        rank_mat.to_csv(Path(OUTPUT_DIR)/"wilcoxon_positive_rank.csv")
        rej_map.to_csv(Path(OUTPUT_DIR)/"wilcoxon_rejection_map.csv")
        print("[SAVE] wilcoxon_positive_rank.csv, wilcoxon_rejection_map.csv")
        return rank_mat, rej_map
    _ = wilcoxon_table(metric="AUC")

    def plot_roc_grid(roc_a, roc_b, title_a="Ours", title_b="SGFR", outfile="cmp_roc_grid.png"):
        fig, axs = plt.subplots(2, 2, figsize=(10,8))
        axes = axs.ravel()
        for ax, name in zip(axes, ["LR","SVM","RF","GBC"]):
            fpr, tpr = roc_a[name]
            ax.plot(fpr, tpr, label=title_a)
            fpr2, tpr2 = roc_b[name]
            ax.plot(fpr2, tpr2, linestyle="--", label=title_b)
            ax.plot([0,1],[0,1], linestyle=":")
            ax.set_title(f"ROC: {name}")
            ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
            ax.legend(loc="lower right", frameon=False)
        fig.tight_layout()
        p = Path(OUTPUT_DIR)/outfile
        fig.savefig(p, dpi=220); plt.close(fig)
        print(f"[SAVE] {p}")
    plot_roc_grid(roc_ours, roc_sgfr, outfile="cmp_roc_grid.png")

else:
    print("[INFO] BINARY_OUTCOME=False → Skipping SGFR comparative classification panel.")

# ============================================================
# (Next) Step 4: One-Step Backtracking Counterfactuals
#   — Place your counterfactual planner here —
# ============================================================


In [ ]:
# === View artifacts as tables + graphs ===
import os, json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# 1) Auto-detect output directory
CANDS = ["../data/processed/mental_outputs", "data/processed/mental_outputs"]
OUT_DIR = next((p for p in CANDS if os.path.isdir(p)), None)
if OUT_DIR is None:
    raise FileNotFoundError("Couldn't find mental_outputs. Run the pipeline first.")
print("Using output dir:", OUT_DIR)

# 2) Load artifacts
mapping_path = Path(OUT_DIR)/"domain_mapping.json"
rank_path    = Path(OUT_DIR)/"groupwise_ranked_features_full.csv"
prune_path   = Path(OUT_DIR)/"groupwise_pruned_features.csv"
domimp_path  = Path(OUT_DIR)/"domain_importance_ebm.csv"
gapath       = Path(OUT_DIR)/"ga_domain_selection.json"
prior_path   = Path(OUT_DIR)/"domain_priority_final.csv"

domain_map = json.load(open(mapping_path))
map_df = pd.DataFrame(list(domain_map.items()), columns=["module","domain"])

rank_df   = pd.read_csv(rank_path)
pruned_df = pd.read_csv(prune_path)
dom_imp   = pd.read_csv(domimp_path)
priority  = pd.read_csv(prior_path)

ga = json.load(open(gapath))
ga_meta = pd.DataFrame([{
    "fitness": ga.get("fitness"),
    "r2": ga.get("r2"),
    "mae": ga.get("mae"),
    "n_features": ga.get("n_features")
}])
ga_sel = pd.DataFrame({"domain": ga.get("domain_list", [])})
ga_sel["selected_by_GA"] = ga_sel["domain"].isin(ga.get("best_domains", [])).astype(int)

# 3) Show tables (previews)
print("\n— Domain mapping (expert-defined) —")
display(map_df.head(30))

print("\n— Groupwise ranked features (full, top by causal score) —")
display(rank_df.sort_values("shap_importance", ascending=False).head(25))

print("\n— Groupwise pruned features (threshold + Top-K) —")
display(pruned_df.sort_values(["domain","shap_importance"], ascending=[True,False]).head(25))

print("\n— Domain importance (EBM permutation-aggregate) —")
display(dom_imp.sort_values("perm_importance", ascending=False))

print("\n— GA domain selection (meta) —")
display(ga_meta)
print("\n— GA selected domains (flag) —")
display(ga_sel)

print("\n— Final domain priority table —")
display(priority.sort_values("perm_importance", ascending=False))

# 4) Graphs (saved + shown). One plot per figure; no custom colors/styles.
viz_dir = Path(OUT_DIR)
viz_dir.mkdir(exist_ok=True, parents=True)

# (A) Top 20 pruned features by causal importance
tp = pruned_df.sort_values("shap_importance", ascending=False).head(20).copy()
tp["label"] = tp["feature_base"] + " [" + tp["domain"] + "]"
plt.figure()
plt.barh(tp["label"][::-1], tp["shap_importance"][::-1])
plt.xlabel("Causal SHAP importance")
plt.ylabel("Feature [domain]")
plt.title("Top 20 Pruned Features by Causal Importance")
plt.tight_layout()
plt.savefig(viz_dir/"viz_top_pruned_features.png", dpi=200)
plt.show()

# (B) Domain contribution from EBM (permutation importance)
di = dom_imp.sort_values("perm_importance", ascending=False)
plt.figure()
plt.barh(di["domain"][::-1], di["perm_importance"][::-1])
plt.xlabel("Permutation importance (EBM)")
plt.ylabel("Domain")
plt.title("Domain Contribution (EBM)")
plt.tight_layout()
plt.savefig(viz_dir/"viz_domain_importance_ebm.png", dpi=200)
plt.show()

# (C) Final domain priority (EBM importance + GA selection flag)
dp = priority.sort_values("perm_importance", ascending=False).copy()
dp["label"] = dp["domain"] + dp["selected_by_GA"].map({1:" ★", 0:""})
plt.figure()
plt.barh(dp["label"][::-1], dp["perm_importance"][::-1])
plt.xlabel("Optimized domain score")
plt.ylabel("Domain (★ = GA-selected)")
plt.title("Domain Priority (EBM importance + GA selection)")
plt.tight_layout()
plt.savefig(viz_dir/"viz_domain_priority_final.png", dpi=200)
plt.show()

print("\nSaved figures to:", viz_dir)
